# Comparative SHAP Analysis Across Experiment Conditions

This notebook starts the implementation for a report-ready comparison of SHAP behaviour across all FAST experiment conditions and subjects.

## Goals
- Recompute or load cached SHAP summaries per run and subject.
- Test whether attribution is over-concentrated in the delta band.
- Test whether attribution is concentrated in non-language zones more than language-related zones.
- Compare attribution drift across subjects and experiment conditions.
- Produce tables and figures that can be reused in a Markdown report.

## 0. Set Up Environment and Dependencies

This section prepares paths, imports the comparative analysis helpers, and reports the active device. GPU is only required for SHAP recomputation cells.

In [5]:
from __future__ import annotations

import json
import gc
import warnings
from datetime import datetime
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import torch
from IPython.display import display

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
RESULTS_ROOT = PROJECT_ROOT / "results"
OUTPUT_ROOT = RESULTS_ROOT / "shap_condition_comparison"
CACHE_ROOT = OUTPUT_ROOT / "cache"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
TABLE_ROOT = OUTPUT_ROOT / "tables"

for folder in [OUTPUT_ROOT, CACHE_ROOT, FIGURE_ROOT, TABLE_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT / "src") not in __import__("sys").path:
    __import__("sys").path.insert(0, str(PROJECT_ROOT / "src"))

from fast.analysis import (
    DEFAULT_FOCUS_RUNS,
    compare_matched_runs,
    build_subject_summary_tables,
    compute_shap_values,
    default_experiments,
    default_model_config,
    load_experiment_performance_table,
    load_model,
    load_subject_data,
)
from fast.data import SUBJECTS, Zones

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Project root: /home/kay/FAST
Device: cuda
CUDA device: NVIDIA GeForce RTX 4090


## 1. Define Configuration and Inputs

The notebook keeps all 11 runs available, but the main narrative is centered on a smaller set of diagnostically important runs.

The cache directory stores per-run and per-subject summary tables so expensive SHAP recomputation can be skipped on later runs.

In [6]:
SEED = 42
SFREQ = 250
N_BG = 100
N_EXPLAIN = 50
FORCE_RECOMPUTE = False
REQUIRE_CUDA_FOR_SHAP = True
VERBOSE_BUILD = True
TARGET_RUNS = list(default_experiments(PROJECT_ROOT).keys())
FOCUS_RUNS = list(DEFAULT_FOCUS_RUNS)
TARGET_SUBJECTS = list(SUBJECTS)
SMOKE_TEST_SUBJECTS = ["01", "02", "03"]
SMOKE_TEST_RUNS = ["run1_original", "run2_condB", "run11_highpass4"]

EXPERIMENTS = default_experiments(PROJECT_ROOT)
MODEL_CONFIG = default_model_config(sfreq=SFREQ)
PERFORMANCE_CSV = RESULTS_ROOT / "condition_experiments" / "comparison_summary.csv"
EXISTING_SHAP_SUMMARY_CSV = RESULTS_ROOT / "shap_analysis" / "shap_summary.csv"
REPORT_PATH = PROJECT_ROOT / "docs" / "shap-condition-comparison-report.md"

print(f"Runs available: {len(TARGET_RUNS)}")
print(f"Focus runs: {FOCUS_RUNS}")
print(f"Subjects available: {TARGET_SUBJECTS}")
print(f"Cache root: {CACHE_ROOT}")
print(f"Verbose build logging: {VERBOSE_BUILD}")

Runs available: 11
Focus runs: ['run1_original', 'run2_condB', 'run3_condC', 'run6_A_B_C', 'run7_B_C', 'run9_augment_no_es', 'run11_highpass4']
Subjects available: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15']
Cache root: /home/kay/FAST/results/shap_condition_comparison/cache
Verbose build logging: True


## 2. Implement Core Functions

These notebook-local helpers handle cache I/O, controlled SHAP recomputation, and loading tidy tables for later analysis and plotting.

In [7]:
TABLE_NAMES = [
    "class_summary",
    "zone_summary",
    "zone_group_summary",
    "band_summary",
    "zone_time",
    "band_time",
]


def log_step(message: str) -> None:
    if VERBOSE_BUILD:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")


def format_duration(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes, rem = divmod(seconds, 60)
    if minutes < 60:
        return f"{int(minutes)}m {rem:.1f}s"
    hours, minutes = divmod(minutes, 60)
    return f"{int(hours)}h {int(minutes)}m {rem:.1f}s"


def cache_dir_for(run_key: str, sid: str) -> Path:
    return CACHE_ROOT / run_key / f"sub-{sid}"


def cache_files_exist(run_key: str, sid: str) -> bool:
    subject_dir = cache_dir_for(run_key, sid)
    exists = subject_dir.exists() and all((subject_dir / f"{name}.csv").exists() for name in TABLE_NAMES)
    if exists:
        log_step(f"Cache hit for {run_key} | sub-{sid} at {subject_dir}")
    return exists


def write_summary_tables(run_key: str, sid: str, tables: dict[str, pd.DataFrame], metadata: dict[str, object]) -> None:
    subject_dir = cache_dir_for(run_key, sid)
    subject_dir.mkdir(parents=True, exist_ok=True)
    for name, frame in tables.items():
        frame.to_csv(subject_dir / f"{name}.csv", index=False)
    with open(subject_dir / "metadata.json", "w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2)
    log_step(f"Wrote cache tables for {run_key} | sub-{sid} to {subject_dir}")


def read_summary_tables(run_key: str, sid: str) -> dict[str, pd.DataFrame]:
    subject_dir = cache_dir_for(run_key, sid)
    tables = {}
    for name in TABLE_NAMES:
        path = subject_dir / f"{name}.csv"
        tables[name] = pd.read_csv(path) if path.exists() else pd.DataFrame()
    log_step(f"Loaded cached tables for {run_key} | sub-{sid}")
    return tables


def build_subject_cache(
    run_key: str,
    sid: str,
    *,
    n_bg: int = N_BG,
    n_explain: int = N_EXPLAIN,
    run_index: int | None = None,
    total_runs: int | None = None,
    subject_index: int | None = None,
    total_subjects: int | None = None,
) -> dict[str, pd.DataFrame]:
    if REQUIRE_CUDA_FOR_SHAP and not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for SHAP recomputation in this notebook.")

    prefix_parts = []
    if run_index is not None and total_runs is not None:
        prefix_parts.append(f"run {run_index}/{total_runs}")
    if subject_index is not None and total_subjects is not None:
        prefix_parts.append(f"subject {subject_index}/{total_subjects}")
    prefix = f"[{', '.join(prefix_parts)}] " if prefix_parts else ""

    experiment = EXPERIMENTS[run_key]
    checkpoint_path = RESULTS_ROOT / "condition_experiments" / experiment.results_dir_name / f"sub-{sid}" / experiment.checkpoint_name
    log_step(f"{prefix}Starting build for {experiment.display_name} | sub-{sid}")
    log_step(f"{prefix}Checkpoint: {checkpoint_path}")

    subject_timer = perf_counter()

    stage_timer = perf_counter()
    model = load_model(checkpoint_path, MODEL_CONFIG, DEVICE)
    if model is None:
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    log_step(f"{prefix}Loaded model in {format_duration(perf_counter() - stage_timer)}")

    stage_timer = perf_counter()
    x_bg, x_explain, y_explain = load_subject_data(
        experiment.h5_paths,
        sid,
        n_bg=n_bg,
        n_explain=n_explain,
        seed=SEED,
    )
    log_step(
        f"{prefix}Loaded data in {format_duration(perf_counter() - stage_timer)} | "
        f"background={len(x_bg)} explain={len(x_explain)}"
    )

    stage_timer = perf_counter()
    with torch.no_grad():
        logits = model(x_explain.to(DEVICE))
        y_pred = torch.argmax(logits, dim=1).cpu().numpy()
    log_step(f"{prefix}Computed predictions in {format_duration(perf_counter() - stage_timer)}")

    stage_timer = perf_counter()
    log_step(f"{prefix}Starting SHAP computation for {experiment.display_name} | sub-{sid}")
    shap_values = compute_shap_values(model, x_bg, x_explain, device=DEVICE)
    log_step(f"{prefix}Computed SHAP values in {format_duration(perf_counter() - stage_timer)}")

    stage_timer = perf_counter()
    tables = build_subject_summary_tables(
        run_key=run_key,
        experiment=experiment,
        sid=sid,
        shap_values=shap_values,
        x_explain=x_explain.cpu().numpy(),
        y_true=y_explain.cpu().numpy(),
        y_pred=y_pred,
    )
    table_shapes = {name: frame.shape for name, frame in tables.items()}
    log_step(f"{prefix}Built summary tables in {format_duration(perf_counter() - stage_timer)} | {table_shapes}")

    metadata = {
        "run_key": run_key,
        "experiment": experiment.display_name,
        "subject": sid,
        "device": str(DEVICE),
        "n_background": int(len(x_bg)),
        "n_explain": int(len(x_explain)),
        "checkpoint": str(checkpoint_path),
        "built_at": datetime.now().isoformat(timespec="seconds"),
    }
    stage_timer = perf_counter()
    write_summary_tables(run_key, sid, tables, metadata)
    log_step(f"{prefix}Finished writing cache in {format_duration(perf_counter() - stage_timer)}")

    del model, x_bg, x_explain, y_explain, logits, shap_values
    torch.cuda.empty_cache()
    gc.collect()
    log_step(f"{prefix}Completed {experiment.display_name} | sub-{sid} in {format_duration(perf_counter() - subject_timer)}")
    return tables


def load_or_build_cache(run_keys: list[str], subjects: list[str], *, force: bool = FORCE_RECOMPUTE) -> dict[tuple[str, str], dict[str, pd.DataFrame]]:
    cache = {}
    total_runs = len(run_keys)
    total_subjects = len(subjects)
    total_tasks = total_runs * total_subjects
    completed_tasks = 0
    build_timer = perf_counter()
    log_step(
        f"Starting cache pass | mode runs={total_runs} subjects={total_subjects} total_tasks={total_tasks} force={force}"
    )

    for run_position, run_key in enumerate(run_keys, start=1):
        experiment = EXPERIMENTS[run_key]
        run_timer = perf_counter()
        log_step(f"Entering {experiment.display_name} ({run_position}/{total_runs})")
        for subject_position, sid in enumerate(subjects, start=1):
            completed_tasks += 1
            progress_prefix = f"task {completed_tasks}/{total_tasks}"
            if force or not cache_files_exist(run_key, sid):
                log_step(f"{progress_prefix}: building {run_key} | sub-{sid}")
                try:
                    cache[(run_key, sid)] = build_subject_cache(
                        run_key,
                        sid,
                        run_index=run_position,
                        total_runs=total_runs,
                        subject_index=subject_position,
                        total_subjects=total_subjects,
                    )
                except Exception as exc:
                    log_step(f"{progress_prefix}: failed for {run_key} | sub-{sid}: {exc}")
                    raise
            else:
                log_step(f"{progress_prefix}: using cached outputs for {run_key} | sub-{sid}")
                cache[(run_key, sid)] = read_summary_tables(run_key, sid)
        log_step(f"Finished {experiment.display_name} in {format_duration(perf_counter() - run_timer)}")

    log_step(f"Completed cache pass in {format_duration(perf_counter() - build_timer)}")
    return cache


def stack_cached_table(cache: dict[tuple[str, str], dict[str, pd.DataFrame]], table_name: str) -> pd.DataFrame:
    frames = [tables[table_name] for tables in cache.values() if table_name in tables and not tables[table_name].empty]
    stacked = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    log_step(f"Stacked {table_name}: {stacked.shape}")
    return stacked

## 3. Compose the Main Execution Flow

Start by loading the existing experiment-level summaries, then choose whether to use existing per-subject caches or build a small smoke-test cache before scaling out to all runs and subjects.

In [ ]:
performance_summary = load_experiment_performance_table(PERFORMANCE_CSV)
existing_shap_summary = pd.read_csv(EXISTING_SHAP_SUMMARY_CSV)

log_step("Loaded performance and existing SHAP summaries")
print("Performance summary:")
display(performance_summary)
print("Existing SHAP summary:")
display(existing_shap_summary)

# RUN_MODE = "load-existing"
RUN_MODE = "smoke-test-build"
# RUN_MODE = "full-build"
log_step(f"RUN_MODE set to {RUN_MODE}")

if RUN_MODE == "load-existing":
    selected_runs = [run_key for run_key in TARGET_RUNS if (CACHE_ROOT / run_key).exists()]
    selected_subjects = TARGET_SUBJECTS
elif RUN_MODE == "smoke-test-build":
    selected_runs = SMOKE_TEST_RUNS
    selected_subjects = SMOKE_TEST_SUBJECTS
else:
    selected_runs = TARGET_RUNS
    selected_subjects = TARGET_SUBJECTS

log_step(f"Selected runs ({len(selected_runs)}): {selected_runs}")
log_step(f"Selected subjects ({len(selected_subjects)}): {selected_subjects}")
log_step(f"Expected build tasks: {len(selected_runs) * len(selected_subjects)}")

analysis_timer = perf_counter()
analysis_cache = load_or_build_cache(
    selected_runs,
    selected_subjects,
    force=FORCE_RECOMPUTE if RUN_MODE != "load-existing" else False,
)
log_step(f"Finished load_or_build_cache in {format_duration(perf_counter() - analysis_timer)}")

class_summary = stack_cached_table(analysis_cache, "class_summary")
zone_summary = stack_cached_table(analysis_cache, "zone_summary")
zone_group_summary = stack_cached_table(analysis_cache, "zone_group_summary")
band_summary = stack_cached_table(analysis_cache, "band_summary")
zone_time_summary = stack_cached_table(analysis_cache, "zone_time")
band_time_summary = stack_cached_table(analysis_cache, "band_time")

log_step("Finished stacking cached tables")
for name, frame in {
    "class_summary": class_summary,
    "zone_summary": zone_summary,
    "zone_group_summary": zone_group_summary,
    "band_summary": band_summary,
    "zone_time_summary": zone_time_summary,
    "band_time_summary": band_time_summary,
}.items():
    print(f"{name}: {frame.shape}")

[12:07:04] Loaded performance and existing SHAP summaries
Performance summary:


,Experiment,Augmented,Trials/Subject,Note,Mean_Val_Acc,Std_Val_Acc,Mean_Test_Acc,Std_Test_Acc,Mean_Test_F1,Std_Test_F1,run_key
0,Run 1: Original Data (A),No,350,1x data,0.736190,0.050325,0.634667,0.082277,0.631096,0.082657,run1_original
1,Run 2: Condition B (No Delta),No,350,1x data,0.415238,0.061437,0.249333,0.062731,0.178999,0.072315,run2_condB
2,Run 3: Condition C (No Artifacts),No,350,1x data,0.426667,0.059851,0.233333,0.042538,0.176499,0.056241,run3_condC
3,Run 4: Original + Condition B,No,700,2x data,0.606667,0.036965,0.644000,0.060095,0.642063,0.059910,run4_A_B
4,Run 5: Original + Condition C,No,700,2x data,0.618095,0.041638,0.650667,0.080309,0.645394,0.081064,run5_A_C
5,Run 6: Original + Cond B + Cond C,No,1050,3x data,0.737778,0.073407,0.653333,0.075467,0.649558,0.075675,run6_A_B_C
6,Run 7: Condition B + Condition C,No,700,2x data,0.716667,0.122910,0.272000,0.055446,0.208285,0.072100,run7_B_C
7,Run 8: Augmentations + Cool Down,Yes,350,Stochastic augmentation,0.724762,0.068398,0.620000,0.078194,0.616941,0.076616,run8_augment
8,Run 9: Augmentations (No Early Stopping),Yes,350,Stochastic augmentation,0.760952,0.052480,0.674667,0.081229,0.673742,0.084414,run9_augment_no_es
9,Run 10: Kaggle Replication,Yes,350,Kaggle replication (no CV),0.733333,0.084063,0.733333,0.084063,0.731914,0.085642,run10_kaggle


Existing SHAP summary:


,Experiment,Mean |SHAP|,Std |SHAP|,Min |SHAP|,Max |SHAP|,Top Zone,N Subjects
0,Run 1: Original (A),0.000538,0.000162,0.000274,0.000923,Pre-frontal,15
1,Run 2: Cond B (No Delta),0.000663,0.000571,0.000008,0.001724,Central,15
2,Run 3: Cond C (No Artifacts),0.000845,0.000610,0.000017,0.002064,Occipital,15
3,Run 4: A + B,0.001199,0.000295,0.000685,0.001779,Occipital,15
4,Run 5: A + C,0.001327,0.000354,0.000710,0.002087,Occipital,15
5,Run 6: A + B + C,0.002438,0.000539,0.001208,0.003540,Occipital,15
6,Run 7: B + C,0.002808,0.000692,0.001762,0.004187,Occipital,15
7,Run 8: Augmented + ES,0.000402,0.000057,0.000318,0.000549,Pre-frontal,15
8,Run 9: Augmented No ES,0.000430,0.000066,0.000318,0.000576,Pre-frontal,15
9,Run 10: Kaggle Replication,0.000406,0.000062,0.000318,0.000541,Pre-frontal,15


[12:07:04] RUN_MODE set to full-build
[12:07:04] Selected runs (11): ['run1_original', 'run2_condB', 'run3_condC', 'run4_A_B', 'run5_A_C', 'run6_A_B_C', 'run7_B_C', 'run8_augment', 'run9_augment_no_es', 'run10_kaggle', 'run11_highpass4']
[12:07:04] Selected subjects (15): ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15']
[12:07:04] Expected build tasks: 165
[12:07:04] Starting cache pass | mode runs=11 subjects=15 total_tasks=165 force=False
[12:07:04] Entering Run 1: Original (A) (1/11)
[12:07:04] task 1/165: building run1_original | sub-01
[12:07:04] [run 1/11, subject 1/15] Starting build for Run 1: Original (A) | sub-01
[12:07:04] [run 1/11, subject 1/15] Checkpoint: /home/kay/FAST/results/condition_experiments/run1_original/sub-01/best_subject.pth
[12:07:04] [run 1/11, subject 1/15] Loaded model in 0.0s
[12:07:04] [run 1/11, subject 1/15] Loaded data in 0.2s | background=100 explain=50
[12:07:04] [run 1/11, subject 1/15] Computed prediction

KeyboardInterrupt: 

## 4. Validate Outputs and Handle Errors

These helpers normalize summaries into comparable shares, run matched-subject contrasts, and surface unstable subjects whose attribution maps shift strongly across conditions.

In [ ]:
def require_non_empty(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    if frame.empty:
        raise ValueError(f"{name} is empty. Build or load cache data first.")
    return frame


def prepare_band_share_table(band_frame: pd.DataFrame) -> pd.DataFrame:
    log_step("Preparing band-share table")
    band_frame = require_non_empty(band_frame, "band_summary").copy()
    keys = ["run_key", "experiment", "subject", "class_name"]
    totals = band_frame.groupby(keys)["band_importance_share"].transform("sum")
    band_frame["band_share"] = np.where(totals > 0, band_frame["band_importance_share"] / totals, np.nan)
    log_step(f"Prepared band-share table: {band_frame.shape}")
    return band_frame


def prepare_zone_group_share_table(zone_group_frame: pd.DataFrame) -> pd.DataFrame:
    log_step("Preparing zone-group-share table")
    zone_group_frame = require_non_empty(zone_group_frame, "zone_group_summary").copy()
    keys = ["run_key", "experiment", "subject", "class_name"]
    totals = zone_group_frame.groupby(keys)["mean_abs_shap"].transform("sum")
    zone_group_frame["zone_group_share"] = np.where(totals > 0, zone_group_frame["mean_abs_shap"] / totals, np.nan)
    log_step(f"Prepared zone-group-share table: {zone_group_frame.shape}")
    return zone_group_frame


def delta_contrast_table(band_frame: pd.DataFrame, run_pairs: list[tuple[str, str]]) -> pd.DataFrame:
    log_step(f"Computing delta contrast table for {len(run_pairs)} run pairs")
    delta_frame = prepare_band_share_table(band_frame)
    delta_frame = delta_frame.loc[delta_frame["band"] == "Delta"]
    rows = [compare_matched_runs(delta_frame, run_a=run_a, run_b=run_b, value_col="band_share") for run_a, run_b in run_pairs]
    results = pd.DataFrame(rows)
    log_step(f"Delta contrast table ready: {results.shape}")
    return results


def zone_group_contrast_table(zone_group_frame: pd.DataFrame, run_pairs: list[tuple[str, str]]) -> pd.DataFrame:
    log_step(f"Computing zone-group contrast table for {len(run_pairs)} run pairs")
    zone_group_frame = prepare_zone_group_share_table(zone_group_frame)
    rows = []
    for zone_group in ["Language", "Motor-Speech", "Non-Language"]:
        subset = zone_group_frame.loc[zone_group_frame["zone_group"] == zone_group]
        for run_a, run_b in run_pairs:
            result = compare_matched_runs(subset, run_a=run_a, run_b=run_b, value_col="zone_group_share")
            result["zone_group"] = zone_group
            rows.append(result)
    results = pd.DataFrame(rows)
    log_step(f"Zone-group contrast table ready: {results.shape}")
    return results


def subject_zone_profile(zone_frame: pd.DataFrame) -> pd.DataFrame:
    log_step("Building subject zone profiles")
    zone_frame = require_non_empty(zone_frame, "zone_summary")
    profile = (
        zone_frame.groupby(["run_key", "subject", "zone"], as_index=False)["mean_abs_shap"]
        .mean()
        .pivot_table(index=["run_key", "subject"], columns="zone", values="mean_abs_shap")
        .fillna(0.0)
        .reset_index()
    )
    log_step(f"Subject zone profiles ready: {profile.shape}")
    return profile


def subject_stability_table(zone_frame: pd.DataFrame, baseline_run: str = "run1_original") -> pd.DataFrame:
    log_step(f"Computing subject stability against baseline {baseline_run}")
    profile = subject_zone_profile(zone_frame)
    zone_cols = [col for col in profile.columns if col not in {"run_key", "subject"}]
    baseline = profile.loc[profile["run_key"] == baseline_run].set_index("subject")
    rows = []
    for _, row in profile.iterrows():
        subject = row["subject"]
        run_key = row["run_key"]
        if run_key == baseline_run or subject not in baseline.index:
            continue
        base_values = baseline.loc[subject, zone_cols].to_numpy(dtype=float)
        run_values = row[zone_cols].to_numpy(dtype=float)
        corr = np.corrcoef(base_values, run_values)[0, 1] if np.std(base_values) > 0 and np.std(run_values) > 0 else np.nan
        drift = float(np.mean(np.abs(run_values - base_values)))
        rows.append({
            "subject": subject,
            "run_key": run_key,
            "baseline_run": baseline_run,
            "zone_profile_corr": corr,
            "zone_profile_l1_drift": drift,
        })
    results = pd.DataFrame(rows)
    log_step(f"Subject stability table ready: {results.shape}")
    return results

## 5. Run Sample Cases

The next cells compute the first report-ready tables and figures using either the smoke-test cache or any previously saved full cache.

In [ ]:
def plot_metric_heatmap(frame: pd.DataFrame, *, index: str, columns: str, values: str, title: str, output_name: str) -> pd.DataFrame:
    log_step(f"Rendering heatmap {output_name} from {values}")
    pivot = frame.pivot_table(index=index, columns=columns, values=values, aggfunc="mean")
    fig, ax = plt.subplots(figsize=(12, 6))
    image = ax.imshow(pivot.to_numpy(), aspect="auto", cmap="YlOrRd", interpolation="nearest")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(title)
    plt.colorbar(image, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.savefig(FIGURE_ROOT / output_name, dpi=200)
    plt.close(fig)
    log_step(f"Saved figure to {FIGURE_ROOT / output_name}")
    return pivot


def plot_contrast_bars(frame: pd.DataFrame, label_col: str, value_col: str, title: str, output_name: str) -> None:
    log_step(f"Rendering contrast bar chart {output_name}")
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(frame[label_col], frame[value_col], color="#1f77b4")
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel(value_col)
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.savefig(FIGURE_ROOT / output_name, dpi=200)
    plt.close(fig)
    log_step(f"Saved figure to {FIGURE_ROOT / output_name}")


run_pairs = [
    ("run1_original", "run2_condB"),
    ("run1_original", "run3_condC"),
    ("run1_original", "run11_highpass4"),
    ("run1_original", "run6_A_B_C"),
]
log_step(f"Using run pairs: {run_pairs}")

analysis_stage_timer = perf_counter()
delta_results = delta_contrast_table(band_summary, run_pairs) if not band_summary.empty else pd.DataFrame()
zone_group_results = zone_group_contrast_table(zone_group_summary, run_pairs) if not zone_group_summary.empty else pd.DataFrame()
stability_results = subject_stability_table(zone_summary) if not zone_summary.empty else pd.DataFrame()
log_step(f"Prepared analysis result tables in {format_duration(perf_counter() - analysis_stage_timer)}")

if not band_summary.empty:
    band_share_table = prepare_band_share_table(band_summary)
    delta_share_table = band_share_table.loc[band_share_table["band"] == "Delta"]
    delta_heatmap = plot_metric_heatmap(
        delta_share_table,
        index="subject",
        columns="run_key",
        values="band_share",
        title="Delta SHAP Share by Subject and Run",
        output_name="delta_share_heatmap.png",
    )
    display(delta_heatmap)

if not zone_group_summary.empty:
    zone_share_table = prepare_zone_group_share_table(zone_group_summary)
    non_language = zone_share_table.loc[zone_share_table["zone_group"] == "Non-Language"]
    non_language_heatmap = plot_metric_heatmap(
        non_language,
        index="subject",
        columns="run_key",
        values="zone_group_share",
        title="Non-Language Zone Share by Subject and Run",
        output_name="non_language_share_heatmap.png",
    )
    display(non_language_heatmap)

if not delta_results.empty:
    delta_results["contrast"] = delta_results["run_a"] + " -> " + delta_results["run_b"]
    plot_contrast_bars(delta_results, "contrast", "mean_diff", "Matched Delta-Share Contrasts", "delta_contrast_bars.png")
    display(delta_results)

if not zone_group_results.empty:
    display(zone_group_results)

if not stability_results.empty:
    display(stability_results.sort_values("zone_profile_corr"))
    stability_results.to_csv(TABLE_ROOT / "subject_stability.csv", index=False)
    log_step(f"Saved subject stability table to {TABLE_ROOT / 'subject_stability.csv'}")

if not delta_results.empty:
    delta_results.to_csv(TABLE_ROOT / "delta_contrasts.csv", index=False)
    log_step(f"Saved delta contrast table to {TABLE_ROOT / 'delta_contrasts.csv'}")
if not zone_group_results.empty:
    zone_group_results.to_csv(TABLE_ROOT / "zone_group_contrasts.csv", index=False)
    log_step(f"Saved zone-group contrast table to {TABLE_ROOT / 'zone_group_contrasts.csv'}")

## 6. Add Basic Tests

These checks are intentionally lightweight. They verify that the notebook is using the expected run aliases, zone groups, and summary-table shapes before a full report is written.

In [ ]:
log_step("Starting report export and notebook checks")
report_sections = [
    "# Comparative SHAP Condition Report",
    "",
    "## Execution Summary",
    f"- Device used for this session: {DEVICE}",
    f"- Selected runs: {selected_runs}",
    f"- Selected subjects: {selected_subjects}",
    "",
    "## Key Tables",
]

if not delta_results.empty:
    report_sections.append("### Delta Share Contrasts")
    report_sections.append(delta_results.to_markdown(index=False))
    report_sections.append("")

if not zone_group_results.empty:
    report_sections.append("### Zone Group Contrasts")
    report_sections.append(zone_group_results.to_markdown(index=False))
    report_sections.append("")

if not stability_results.empty:
    report_sections.append("### Subject Stability")
    report_sections.append(stability_results.sort_values("zone_profile_corr").head(15).to_markdown(index=False))
    report_sections.append("")

REPORT_PATH.write_text("\n".join(report_sections), encoding="utf-8")
log_step(f"Wrote report draft to {REPORT_PATH}")

assert set(FOCUS_RUNS).issubset(set(EXPERIMENTS.keys()))
assert len(Zones) == 8
assert performance_summary["run_key"].nunique() == 11
if not zone_group_summary.empty:
    assert set(zone_group_summary["zone_group"].dropna().unique()) <= {"Language", "Motor-Speech", "Non-Language"}
if not delta_results.empty:
    assert {"run_a", "run_b", "mean_diff", "p_value"}.issubset(delta_results.columns)
log_step("Basic notebook checks passed")
print("Basic notebook checks passed.")

Wrote report draft to /home/kay/FAST/docs/shap-condition-comparison-report.md
Basic notebook checks passed.
